<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

This is a scoring task. I am framing refresh / content opportunity scoring as a page-level priority score: for each content item, the model will estimate how promising it is as a refresh candidate, and the editorial team can then rank pages for review. I choose scoring rather than a hard classification because the action is an ordered queue, not a simple yes/no decision.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Scoring task framed for refresh opportunity ranking.")

Scoring task framed for refresh opportunity ranking.


## 2. Target or proxy

The target I would predict is a proxy for whether a content page is a real refresh opportunity. In this starter dataset, I can use the observed recent decline signal `trend_direction` as a practical proxy for that idea, because it captures whether the page has recently moved down in a measurable way. I would treat it as a proxy for the first pass and later validate it against a future outcome window.

In [3]:
import pandas as pd
from pathlib import Path

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent.parent

# Load the starter slice for the refresh lane.
df = pd.read_csv(root / 'data' / 'raw' / 'content_refresh_anonymized.csv')

# Build a simple proxy label from the observed decline signal.
proxy = df['trend_direction'].eq('down').fillna(False).astype(int)

print('Proxy label preview:')
print(pd.Series(proxy.head(10).tolist()).to_string(index=False))

Proxy label preview:
1
1
1
0
1
1
1
0
1
1


## 3. Success metric

A defendable metric here is precision@K for the top 100 pages in the review queue. If the model is good, most of the pages near the top of the queue should be genuinely worthwhile refresh opportunities rather than noisy or low-value pages. In plain terms, a higher precision@100 means the editorial team wastes less time on the wrong pages.

In [4]:
lane_df = df[['content_id', 'client_id', 'content_type', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count', 'trend_direction']].copy()
lane_df['refresh_opportunity_proxy'] = proxy

unit_df = lane_df.head(8).copy()
print('Rows in lane slice:', len(lane_df))
print('Proxy-positive rate:', round(lane_df['refresh_opportunity_proxy'].mean(), 3))
display(unit_df)
print('\nOne row = one content item (page) at the snapshot date.')
print('Target column preview:')
print(lane_df[['trend_direction', 'refresh_opportunity_proxy']].head(10).to_string(index=False))

Rows in lane slice: 30000
Proxy-positive rate: 0.542


,content_id,client_id,content_type,content_age_days,impressions_90d,avg_position,ctr,engagement_rate,scroll_rate,word_count,trend_direction,refresh_opportunity_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,187,3803,10.6,0.76,5.88,4.55,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,15320,20.3,0.05,0.00,10.00,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,12581,36.5,0.09,0.00,28.57,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,11751,6.2,0.49,1.28,3.45,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,19140,44.0,0.13,0.00,24.29,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,3970,8.5,0.03,0.00,25.00,3080.0,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,7.0,0.00,0.00,0.00,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,1724,21.2,0.06,3.57,7.14,NaN,stable,0



One row = one content item (page) at the snapshot date.
Target column preview:
trend_direction  refresh_opportunity_proxy
           down                          1
           down                          1
           down                          1
         stable                          0
           down                          1
           down                          1
           down                          1
         stable                          0
           down                          1
           down                          1


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one content page, not one client and not one query. The dataframe above shows a row for each content item with its observed performance signals and the proxy target for whether it looks like a refresh opportunity.

In [5]:
print('Unit of analysis confirmed: one row = one content item (page).')
print('Example row fields:')
print(unit_df.columns.tolist())

Unit of analysis confirmed: one row = one content item (page).
Example row fields:
['content_id', 'client_id', 'content_type', 'content_age_days', 'impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate', 'word_count', 'trend_direction', 'refresh_opportunity_proxy']


## 5. Why ML beats a fixed rule here

A fixed rule like “flag any page with low impressions and a negative trend” is too brittle. The real signal is a mix of weak, interacting patterns: page age, position, engagement, content type, visibility, and client-specific behavior all matter in different combinations. That makes the pattern messy enough that a hand-written rule will miss many worthwhile opportunities and over-flag many low-value ones. ML is useful here because it can learn those interactions from historical examples rather than pretending one threshold is enough.

In [6]:
visible = lane_df['impressions_90d'] >= 500
print('Visible pages:', int(visible.sum()))
print('Visible and proxy-positive:', int(((visible) & (lane_df['refresh_opportunity_proxy'] == 1)).sum()))
print('This supports a ranked action for editorial review rather than a single hard rule.')

Visible pages: 16726
Visible and proxy-positive: 9961
This supports a ranked action for editorial review rather than a single hard rule.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.